# Ordered Logistic Regression Results: Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIRˆ² dataset using the `mlcroissant` library in Python, guided by the Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Access and print metadata summary
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")
print(f"Published: {meta.datePublished}, Version: {meta.version}, Identifier: {meta.identifier}")

## 2. Data Overview
Review available record sets and their field and column information, referencing by `@id`.

In [ ]:
# List available record sets and their properties
if hasattr(meta, 'record_sets'):
    print(f"Total record sets: {len(meta.record_sets)}")
    for rs in meta.record_sets:
        print(f"Record set name: {rs.name} @id: {rs.id}")
        if hasattr(rs, 'fields'):
            print("  Fields:")
            for f in rs.fields:
                print(f"    - {f.name}, @id: {f.id}, datatype: {getattr(f, 'dataType', 'n/a')}")
        if hasattr(rs, 'columns'):
            print("  Columns:")
            for col in rs.columns:
                print(f"    - {col.name}, @id: {col.id}, datatype: {getattr(col, 'dataType', 'n/a')}")
else:
    print("No record sets found in this dataset.")

## 3. Data Extraction
Load data from specific record sets using their `@id`s into Pandas DataFrames.
Replace the list of record set `@id`s with those you wish to extract.

In [ ]:
# -- Inspect record sets
record_set_ids = []
if hasattr(meta, 'record_sets'):
    record_set_ids = [rs.id for rs in meta.record_sets]
else:
    print("No record sets to extract.")

# Extract each record set to a DataFrame
dataframes = {}
for rs_id in record_set_ids:
    # Use @id for referencing the record set
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"Loaded DataFrame for record set @id: {rs_id}, columns: {list(dataframes[rs_id].columns)}")
    else:
        print(f"No records for record set @id: {rs_id}")

# Display columns and preview for the first available DataFrame
if dataframes:
    first_rs_id = list(dataframes.keys())[0]
    print(f"\nPreview of DataFrame for {first_rs_id}:")
    display(dataframes[first_rs_id].head())
else:
    print("No dataframes to preview.")

## 4. Exploratory Data Analysis (EDA)
The following provides examples of filtering and transforming the data using field `@id`s. Replace `<record_set_id>`, `<numeric_field_id>`, and `<group_field_id>` as appropriate for your dataset.

In [ ]:
# Example: Select a numeric field for filtering and normalization
# Replace these with real @id values from the Data Overview above.
record_set_id = None
numeric_field_id = None
group_field_id = None

# Attempt automatic selection for demonstration (selecting first DataFrame with numeric columns)
for rs_id, df in dataframes.items():
    num_cols = df.select_dtypes(include='number').columns.tolist()
    if num_cols:
        record_set_id = rs_id
        numeric_field_id = num_cols[0]
        # Attempt to guess a categorical/group field
        cat_cols = df.select_dtypes(include='object').columns.tolist()
        group_field_id = cat_cols[0] if cat_cols else None
        break

if record_set_id and numeric_field_id:
    df = dataframes[record_set_id]
    threshold = df[numeric_field_id].mean() if df[numeric_field_id].notnull().any() else 0
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records in '{record_set_id}' with '{numeric_field_id}' > mean ({threshold:.2f}):")
    display(filtered_df.head())

    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized '{numeric_field_id}' for filtered records:")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped by '{group_field_id}':")
        display(grouped_df.head())
else:
    print("Could not automatically select record set and numeric field. Please fill in the appropriate @id values.")

## 5. Visualization
Visualize numeric field distributions and relationships between variables.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Basic histogram and boxplot for numeric field
if record_set_id and numeric_field_id and record_set_id in dataframes:
    df = dataframes[record_set_id]
    plt.figure(figsize=(9, 4))
    plt.subplot(1,2,1)
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field_id}")

    plt.subplot(1,2,2)
    sns.boxplot(y=df[numeric_field_id].dropna())
    plt.title(f"Boxplot of {numeric_field_id}")
    plt.tight_layout()
    plt.show()

    # If grouping field exists, show grouped means
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8,4))
        sns.barplot(x=group_field_id, y=numeric_field_id, data=df, estimator='mean', ci=None)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Cannot visualize: record set and field IDs not set.")

## 6. Conclusion
This notebook demonstrated loading FAIRˆ² dataset metadata and records via Croissant schema, reviewing record sets and fields by `@id`, extracting datasets to Pandas DataFrames, running exploratory data analysis (EDA), and visualizing numeric attributes. Replace placeholder IDs with actual ones per your data context for a more granular exploration.